## What the "magic bridge" does in simple terms

The `play(taskCode)` function is a function that connects your notebook code to the browser game window.

In simple terms, it does 4 things:
1. Checks that you are running inside GitHub Codespaces.
2. Builds the public web link to your running app (`shell.html`) on port 8080.
3. Puts your Phaser script (`taskCode`) into that link safely.
4. Shows a button in the notebook so you can click and open the game in a new tab.

So instead of manually finding and building URLs, you just call `play(...)` and it launches your Phaser scene.

In [ ]:
// TEACHER SETUP: THE MAGIC BRIDGE
function play(taskCode) {
    const name = Deno.env.get("CODESPACE_NAME");
    const domain = Deno.env.get("GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN");
    
    if (!name || !domain) {
        return "ERROR: Are you running this in a GitHub Codespace?";
    }

    // Open shell and send code via postMessage to avoid URL-size limits.
    const baseUrl = `https://${name}-8080.${domain}`;
    const shellUrl = `${baseUrl}/shell.html`;
    const shellUrlJson = JSON.stringify(shellUrl);
    const taskCodeJson = JSON.stringify(taskCode);
    
    // Display a launch panel in the notebook
    Deno.jupyter.display({
        "text/html": `
            <div style="padding: 15px; border: 2px solid #4A90E2; border-radius: 8px; background: #f0f7ff; font-family: sans-serif;">
                <p style="margin: 0 0 10px 0; color: #333;">🚀 <b>Phaser Stage Ready!</b></p>
                <button id="open-stage-btn" style="display: inline-block; padding: 8px 16px; background: #4A90E2; color: white; border: none; border-radius: 4px; font-weight: bold; cursor: pointer;">Open Big Screen Window</button>
                <p id="open-stage-status" style="margin: 10px 0 0 0; color: #555; font-size: 12px;"></p>
            </div>
            <script>
                (() => {
                    const btn = document.getElementById("open-stage-btn");
                    const status = document.getElementById("open-stage-status");
                    const url = ${shellUrlJson};
                    const payload = ${taskCodeJson};

                    btn.addEventListener("click", () => {
                        const child = window.open(url, "_blank");
                        if (!child) {
                            status.textContent = "Popup blocked. Please allow popups and try again.";
                            return;
                        }

                        status.textContent = "Opening stage window...";
                        let acked = false;
                        let tries = 0;
                        let retryTimer = null;

                        const cleanup = () => {
                            window.removeEventListener("message", onMessage);
                            if (retryTimer) {
                                clearInterval(retryTimer);
                                retryTimer = null;
                            }
                        };

                        const sendPayload = () => {
                            if (acked) return;
                            tries += 1;
                            try {
                                child.postMessage({ type: "RUN_TASK", code: payload }, "*");
                                status.textContent = "Sending script to stage window... (attempt " + tries + ")";
                            } catch (_e) {
                                status.textContent = "Send failed. Please retry.";
                            }
                        };

                        const onMessage = (event) => {
                            if (!event.data) return;
                            if (event.data.type === "SHELL_READY") {
                                sendPayload();
                                return;
                            }
                            if (event.data.type === "SHELL_ACK") {
                                acked = true;
                                status.textContent = "Script delivered to stage window.";
                                cleanup();
                            }
                        };

                        window.addEventListener("message", onMessage);

                        // Fire once quickly, then retry until ACK or timeout.
                        sendPayload();
                        retryTimer = setInterval(() => {
                            if (acked || tries >= 20) {
                                if (!acked) {
                                    status.textContent = "Delivery timed out. Close stage tab and try again.";
                                }
                                cleanup();
                                return;
                            }
                            sendPayload();
                        }, 300);
                    });
                })();
            </script>
        `
    }, { raw: true });
}

## Bouncing Ball Example Game

Before we can run the Phaser game, we need to start a web server. The "magic bridge" we built allows us to link our notebook code to the browser game window but there is still now web server to make this link. To activate the server. Run the command below in your terminal to start a simple file server that serves your Phaser game:

`deno run --allow-net --allow-read https://deno.land/std@0.190.0/http/file_server.ts --port 8080`

The code below is a simple Phaser game that shows a bouncing ball. You can run the whole cell to see it in action!.

In [2]:
// STUDENT SCRIPT: Bouncing Ball Sanity Check
const sanityCheck = `
    const config = {
        type: Phaser.AUTO,
        parent: 'phaser-game',
        width: 600,
        height: 400,
        backgroundColor: '#ffffff',
        scene: {
            create: function() {
                this.add.text(120, 150, 'SANITY CHECK PASSED', { 
                    fontSize: '32px', fill: '#4A90E2', fontStyle: 'bold' 
                });
                
                const ball = this.add.circle(300, 300, 30, 0x50E3C2);
                
                // Make the ball bounce to prove the engine is running
                this.tweens.add({
                    targets: ball,
                    y: 100,
                    duration: 1000,
                    yoyo: true,
                    loop: -1,
                    ease: 'Sine.easeInOut'
                });
            }
        }
    };
    new Phaser.Game(config);
`;

// Run the magic bridge
play(sanityCheck);

"ERROR: Are you running this in a GitHub Codespace?"